# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/adityabhagwani/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df[feature_names]
y = df["Class"]

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=20, batch_size=8, validation_split=0.2)

Epoch 1/20
13/13 [==============================] - 0s 5ms/step - loss: 0.9445 - accuracy: 0.5354 - val_loss: 0.8323 - val_accuracy: 0.6800
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.6429 - accuracy: 0.9091 - val_loss: 0.5696 - val_accuracy: 0.8800
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4458 - accuracy: 0.9596 - val_loss: 0.4010 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.3150 - accuracy: 0.9899 - val_loss: 0.2886 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2303 - accuracy: 0.9899 - val_loss: 0.2146 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1720 - accuracy: 0.9899 - val_loss: 0.1593 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1312 - accuracy: 0.9899 - val_loss: 0.1266 - val_accuracy: 1.0000
Epoch 8/20
13/13 [==

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 2ms/step - loss: 0.0319 - accuracy: 1.0000
Test Accuracy: 1.0000
2/2 [==============================] - 0s 822us/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [14]:
import os
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"TFLite model size: {size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp79lfl0b7/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp79lfl0b7/assets


TFLite model size: 14.07 KB


2026-06-07 13:01:42.705156: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-06-07 13:01:42.705390: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:01:42.705766: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp79lfl0b7
2026-06-07 13:01:42.706246: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:01:42.706252: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp79lfl0b7
2026-06-07 13:01:42.709744: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-06-07 13:01:42.731982: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp79lfl0b7
2026-06-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [16]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]

def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings
    if quant_type == 'int8':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert and save
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if needed
        if input_details['dtype'] == np.int8:
            scale, zero_point = input_details['quantization']
            sample = (sample / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])

        # Dequantize output if needed
        if output_details['dtype'] == np.int8:
            scale, zero_point = output_details['quantization']
            output = (output.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # Step 4: Report results
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [17]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test, y_test, 'int8',    'model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpn8ihy9zn/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpn8ihy9zn/assets
/Users/adityabhagwani/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-06-07 13:02:51.461782: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-06-07 13:02:51.461796: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:02:51.461910: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpn8ihy9zn
2026-06-07 13:02:51.462352: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:02:51.462356: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwq


INT8 TFLite model size: 5.74 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpaq5ew0be/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpaq5ew0be/assets
2026-06-07 13:02:51.881679: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-06-07 13:02:51.881689: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:02:51.881781: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpaq5ew0be
2026-06-07 13:02:51.882195: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:02:51.882199: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpaq5ew0be
2026-06-07 13:02:51.883236: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-06-07 13:02:51.898699: I tensorflow/cc/saved_model/loader.cc:217] Running initialization


FLOAT16 TFLite model size: 8.95 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp69okvdtm/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp69okvdtm/assets



DYNAMIC TFLite model size: 8.17 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-06-07 13:02:52.108790: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-06-07 13:02:52.108800: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:02:52.108895: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp69okvdtm
2026-06-07 13:02:52.109352: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:02:52.109356: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp69okvdtm
2026-06-07 13:02:52.110466: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-06-07 13:02:52.126089: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmp69okvdtm
2026-06-

## Problem 1 - Part (c)

### Pruning

In [18]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 20
dataset_size = len(X_train)
end_step = (dataset_size // batch_size) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [19]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = tf.keras.Sequential([
    prune_low_magnitude(tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)), pruning_schedule=pruning_schedule),
    prune_low_magnitude(tf.keras.layers.Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(tf.keras.layers.Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule)
])

In [20]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

history_pruned = pruned_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/10
13/13 [==============================] - 1s 6ms/step - loss: 1.1221 - accuracy: 0.5051 - val_loss: 0.8962 - val_accuracy: 0.6400
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7780 - accuracy: 0.7475 - val_loss: 0.6886 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5767 - accuracy: 0.9091 - val_loss: 0.5542 - val_accuracy: 0.8800
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4282 - accuracy: 0.9596 - val_loss: 0.4367 - val_accuracy: 0.8800
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3179 - accuracy: 0.9697 - val_loss: 0.3421 - val_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2360 - accuracy: 0.9697 - val_loss: 0.2758 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1774 - accuracy: 0.9798 - val_loss: 0.2203 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [21]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned)

size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"Pruned TFLite model size: {size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpgq75nlvv/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpgq75nlvv/assets


Pruned TFLite model size: 14.14 KB


2026-06-07 13:04:06.304925: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-06-07 13:04:06.305180: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:04:06.305533: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpgq75nlvv
2026-06-07 13:04:06.305829: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:04:06.305834: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpgq75nlvv
2026-06-07 13:04:06.307897: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-06-07 13:04:06.318067: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpgq75nlvv
2026-06-

In [22]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = np.argmax(stripped_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [23]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [24]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_soft_labels = model.predict(X_train)

4/4 [==============================] - 0s 708us/step


In [25]:
y_train_combined = np.concatenate([y_train, teacher_soft_labels], axis=1)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [26]:
student_model.compile(optimizer='adam', loss=distillation_loss, metrics=['accuracy'])

history_student = student_model.fit(
    X_train, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 0s 5ms/step - loss: 1.3586 - accuracy: 0.3131 - val_loss: 1.3124 - val_accuracy: 0.4000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 1.0975 - accuracy: 0.4949 - val_loss: 1.0917 - val_accuracy: 0.4800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.9133 - accuracy: 0.5859 - val_loss: 0.9249 - val_accuracy: 0.5200
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7847 - accuracy: 0.6364 - val_loss: 0.8135 - val_accuracy: 0.6000
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6914 - accuracy: 0.7475 - val_loss: 0.7223 - val_accuracy: 0.8000
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6072 - accuracy: 0.7980 - val_loss: 0.6508 - val_accuracy: 0.8400
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5382 - accuracy: 0.8182 - val_loss: 0.5824 - val_accuracy: 0.8800
Epoch 8/10
13/13 [==

In [27]:
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd)

size_kb = os.path.getsize("model_kd.tflite") / 1024
print(f"Knowledge Distillation TFLite model size: {size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpx7klpoeo/assets


INFO:tensorflow:Assets written to: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpx7klpoeo/assets
2026-06-07 13:06:49.822397: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


Knowledge Distillation TFLite model size: 6.10 KB


2026-06-07 13:06:49.822693: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-06-07 13:06:49.822990: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpx7klpoeo
2026-06-07 13:06:49.823458: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-06-07 13:06:49.823463: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpx7klpoeo
2026-06-07 13:06:49.825759: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-06-07 13:06:49.845843: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/s5/ywgpwqpx3yq714sfgz4w0t3w0000gn/T/tmpx7klpoeo
2026-06-07 13:06:49.850762: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took

In [28]:
y_pred = np.argmax(student_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      0.86      0.90        21
           2       0.82      1.00      0.90        14

    accuracy                           0.93        54
   macro avg       0.92      0.93      0.93        54
weighted avg       0.93      0.93      0.93        54

Confusion Matrix:
[[18  1  0]
 [ 0 18  3]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
